In [1]:
import os
import random
import tempfile, shutil, os
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
train_data = datasets.MNIST(root="data", train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST(root="data", train=False, download=True, transform=transforms.ToTensor())
train_loader = DataLoader(train_data, batch_size = 64, shuffle = True)
img, label = train_data[0]                   
images, labels = next(iter(train_loader))
test_loader = DataLoader(test_data, batch_size=64)
device = "cuda" if torch.cuda.is_available() else "cpu"
shutil.rmtree(os.path.join(tempfile.gettempdir(), ".tensorboard-info"), ignore_errors=True)

In [2]:
#Feed Forward NN Model
class FFN(nn.Module):
    def __init__(self, n_in, n_out):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(n_in, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, n_out)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

In [3]:
#Convolutional NN Model
class CNN(nn.Module):
    def __init__(self, n_out):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size = 3, padding = 1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size = 3, padding = 1)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(1568, 128)
        self.fc2 = nn.Linear(128, n_out)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [4]:
def train(model, loader, loss_fn, optimizer, epochs, writer = None, test_loader = None):
    step = 0
    for epoch in range(epochs):
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device) 
            out = model(images)
            loss = loss_fn(out, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if writer and step % 50 == 0:
                writer.add_scalar("train_loss", loss.item(), step)
            step += 1
            
        if writer and test_loader:
            writer.add_scalar("test_acc", evaluate(model, test_loader), epoch + 1)


In [5]:
def evaluate(model, loader):
    for test in loader:
        correct = 0
        with torch.no_grad():
            for n, (images, labels) in enumerate (test_loader):
                images, labels = images.to(device), labels.to(device) 
                out = model(images)
                predictions = out.argmax(dim=1)
                correct += (predictions == labels).sum().item()
        
    return round(correct / len(loader.dataset) * 100, 2)

In [6]:
def show_guesses(model, data, n=20):
    for i in range (n):
        x = random.randint(0,len(data) - 1)
        img, label = test_data[x]
        guess = model(img.unsqueeze(0).to(device)).argmax().item()
        plt.imshow(img.squeeze(), cmap="gray")
        plt.title(f"Guess: {guess}, Answer: {label}")
        plt.show()

In [7]:
model = FFN(img.numel(), 10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train(model, train_loader, nn.CrossEntropyLoss(), optimizer, epochs=2)
acc = evaluate(model, test_loader)
print(f"{acc}%")
#show_guesses(model, test_data, n=5)

95.96%


In [8]:
model = CNN(10).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train(model, train_loader, nn.CrossEntropyLoss(), optimizer, epochs=2)
acc = evaluate(model, test_loader)
print(f"{acc}%")
#show_guesses(model, test_data, n=5)

98.66%


In [9]:
import time
shutil.rmtree("runs/compare", ignore_errors=True)
makers = {"FFN": lambda: FFN(img.numel(), 10), "CNN": lambda: CNN(10)}

for name, make in makers.items():
    t = time.time()
    model = make().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    writer = SummaryWriter(f"runs/compare/{name}")
    print(name, "starting training")
    train(model, train_loader, nn.CrossEntropyLoss(), optimizer, 2, writer, test_loader)
    print(name, "trained in", round(time.time() - t), "s")
    writer.close()
    print(name, "closed")

FFN starting training
FFN trained in 452 s
FFN closed
CNN starting training
CNN trained in 481 s
CNN closed


In [10]:
%reload_ext tensorboard
%tensorboard --logdir runs